In [25]:
from typing import Literal

from pydantic import BaseModel

from dreamai.ai import ModelName
from dreamai.dialog import Dialog
from dreamai.dialog_models import SourcedResponse
from dreamai.md_utils import docs_to_md

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
class CreditAgreement(BaseModel):
    aggreement_type: Literal["Investment Grid (IGR)", "Leverage"]
    deal_loan_general_information: str
    tranche_information: str
    interest_and_fees: str
    financial_covenants: str
    pricing_grid_information: str

In [3]:
doc = "/media/hamza/data2/loan1.pdf"

In [4]:
md = docs_to_md(doc)

Processing /media/hamza/data2/loan1.pdf...
[                                        ] (0/16[                                        ] (  1/16[                                        ] (  2/16[                                        ] (  3/16[                                        ] (  4/165[=                                       ] (  5/165[=                                       ] (  6/165[=                                       ] (  7/165[=                                       ] (  8/16[==                                      ] (  9/16[==                                      ] ( 10/16[==                                      ] ( 11/16[==                                      ] ( 12/165[===                                     ] ( 13/1[===                                     ] ( 14/1[===                                     ] ( 15/1[===                                     ] ( 16/16[====                                    ] ( 17/16[====                                    ] ( 18/16[====  

In [23]:
str(md[0].chunks[0])

"name='loan1.pdf' index=0 text='EX-10.1 TERM LOAN CUSIP: 24001QAM3 $150,000,000 TERM LOAN AGREEMENT by and among THE DAYTON POWER & LIGHT COMPANY d/b/a AES Ohio THE LENDERS PARTY HERETO PNC BANK, NATIONAL ASSOCIATION, as Administrative Agent PNC CAPITAL MARKETS LLC, as Bookrunner and Joint Lead Arranger and U.S . BANK NATIONAL ASSOCIATION, as Syndication Agent and Joint Lead Arranger Dated as of August 14, 2024 --- -- **Page** 1.1 Certain Definitions 1 1.2 Construction 22 1.3 Accounting Principles 23 1.4 Term SOFR Notification 23 2 . RESERVED 24 3 . TERM LOAN 24 3.1 Term Loan Commitments 24 3.2 Nature of Lenders’ Obligations with Respect to Term Loan; Repayment Terms . 24 3.3 Use of Proceeds 24 3.4 Defaulting Lenders 24 3.5 Interest Rate Elections; Conversions and Renewals 25 4 . INTEREST RATES 25' metadata={}"

In [26]:
dialog = Dialog(task="loan_task1.txt")

In [27]:
creator, kwargs = dialog.creator_with_kwargs(
    model=ModelName.SONNET, user="\n".join([str(c) for c in md[0].chunks])
)

In [28]:
res = creator.create(
    **kwargs,
    response_model=SourcedResponse,
    validation_context={"documents": [str(c) for c in md[0].chunks]},
)

In [31]:
res2 = creator.create(**kwargs, response_model=CreditAgreement)

In [36]:
dialog2 = Dialog(task="Combine the two responses into a single response.")
creator2, kwargs2 = dialog2.creator_with_kwargs(
    model=ModelName.SONNET,
    user=f"<response1>\n{str(res)}\n</response1>\n\n<response2>\n{res2.model_dump_json()}\n</response2>",  # type:ignore
)
res3 = creator2.create(**kwargs2, response_model=CreditAgreement)

In [39]:
print(str(res))

- The loan agreement is between The Dayton Power and Light Company (the Borrower) and PNC Bank, National Association as Administrative Agent and the Lenders.
- The loan amount is $150,000,000.
- The loan agreement is dated August 14, 2024.
- The loan has a Term SOFR Rate Option and a Base Rate Option for interest.
- The loan maturity date is August 13, 2025.
- The Borrower must maintain a ratio of Consolidated Total Debt to Consolidated Total Capitalization of no greater than 0.67 to 1.00.
- The loan can be prepaid voluntarily with notice to the Administrative Agent.
- Events of Default include failure to pay principal or interest when due, breach of representations or covenants, cross-default on other debt, bankruptcy, and change of control.


In [40]:
res2.model_dump()

{'aggreement_type': 'Leverage',
 'deal_loan_general_information': 'This is a $150,000,000 Term Loan Agreement dated August 14, 2024 between The Dayton Power and Light Company as Borrower, PNC Bank, National Association as Administrative Agent, and various lenders. The loan matures on August 13, 2025.',
 'tranche_information': 'There is a single $150,000,000 term loan tranche.',
 'interest_and_fees': 'The loan bears interest at either a Base Rate Option or Term SOFR Rate Option. The Applicable Margin is 0% for Base Rate loans and 1.05% for Term SOFR Rate loans. There is a SOFR Adjustment of 0.10%.',
 'financial_covenants': 'The Borrower must maintain a ratio of Consolidated Total Debt to Consolidated Total Capitalization not greater than 0.67 to 1.00.',
 'pricing_grid_information': 'There is no pricing grid information provided in the excerpts.'}

In [41]:
res3.model_dump()

{'aggreement_type': 'Leverage',
 'deal_loan_general_information': 'This is a $150,000,000 Term Loan Agreement dated August 14, 2024 between The Dayton Power and Light Company as Borrower, PNC Bank, National Association as Administrative Agent, and various lenders. The loan matures on August 13, 2025. The loan can be prepaid voluntarily with notice to the Administrative Agent. Events of Default include failure to pay principal or interest when due, breach of representations or covenants, cross-default on other debt, bankruptcy, and change of control.',
 'tranche_information': 'There is a single $150,000,000 term loan tranche.',
 'interest_and_fees': 'The loan bears interest at either a Base Rate Option or Term SOFR Rate Option. The Applicable Margin is 0% for Base Rate loans and 1.05% for Term SOFR Rate loans. There is a SOFR Adjustment of 0.10%.',
 'financial_covenants': 'The Borrower must maintain a ratio of Consolidated Total Debt to Consolidated Total Capitalization not greater than

In [2]:
import json

In [3]:
with open("extracted.json", "w") as f:
    json.dump(res3.model_dump(), f)

NameError: name 'res3' is not defined